# WoRMS API -- How It Works

**WoRMS** (World Register of Marine Species) is a freely accessible database that maintains
the accepted list of names for marine organisms. It covers seagrasses, macroalgae, marine
animals, and many other groups. Unlike the Symbiota portals (MyCoPortal, BryophytePortal),
WoRMS provides a proper REST API that returns JSON directly -- no HTML scraping needed.

This notebook walks through the three WoRMS endpoints we need:

| # | Endpoint | What it does |
|---|---|---|
| 1 | `AphiaRecordsByName/{name}` | Look up a species by name; returns the AphiaID and accepted/synonym status |
| 2 | `AphiaSynonymsByAphiaID/{id}` | Get all names linked to an accepted taxon (synonyms at all ranks) |
| 3 | `AphiaClassificationByAphiaID/{id}` | Get the full classification from Kingdom down to Species |

The primary key in WoRMS is called the **AphiaID** -- every taxon record has one.

In [1]:
import requests

BASE_URL = "https://www.marinespecies.org/rest"

---
## 1. `AphiaRecordsByName/{name}` -- Look up a species by name

This endpoint takes a scientific name in the URL path and returns a list of matching records.

Two query parameters we always use:
- **`like=false`** -- require an exact name match (no wildcard prefix search)
- **`marine_only=false`** -- include brackish and non-marine organisms that WoRMS also covers

The response is a list because WoRMS can hold more than one record for the same name string
(for example, when a name appears in both an accepted and an alternative representation).

In [2]:
# Query for Zostera marina (eelgrass), a common marine seagrass
resp = requests.get(
    f"{BASE_URL}/AphiaRecordsByName/Zostera marina",
    params={"like": "false", "marine_only": "false", "offset": 1},
)
resp.json()

[{'AphiaID': 495077,
  'url': 'https://www.marinespecies.org/aphia.php?p=taxdetails&id=495077',
  'scientificname': 'Zostera marina',
  'authority': 'Linnaeus, 1753',
  'status': 'accepted',
  'unacceptreason': None,
  'taxonRankID': 220,
  'rank': 'Species',
  'valid_AphiaID': 495077,
  'valid_name': 'Zostera marina',
  'valid_authority': 'Linnaeus, 1753',
  'parentNameUsageID': 144233,
  'originalNameUsageID': 495077,
  'kingdom': 'Plantae',
  'phylum': 'Tracheophyta',
  'class': 'Magnoliopsida',
  'order': 'Alismatales',
  'family': 'Zosteraceae',
  'genus': 'Zostera',
  'citation': 'Guiry, M.D. & Guiry, G.M. (2026). AlgaeBase. World-wide electronic publication, National University of Ireland, Galway (taxonomic information republished from AlgaeBase with permission of M.D. Guiry). Zostera marina Linnaeus, 1753. Accessed through: World Register of Marine Species at: https://www.marinespecies.org/aphia.php?p=taxdetails&id=495077 on 2026-05-27',
  'lsid': 'urn:lsid:marinespecies.org:ta

The response contains two records for the same name. The key fields to pay attention to:

| Field | Meaning |
|---|---|
| `AphiaID` | The unique ID for this particular record |
| `status` | `"accepted"`, `"unaccepted"`, `"alternative representation"`, etc. |
| `valid_AphiaID` | The AphiaID of the accepted name (equals `AphiaID` when the record is already accepted) |
| `valid_name` | The accepted scientific name |
| `taxonRankID` | Numeric rank: 220 = Species, 240 = Variety, 260 = Forma |

We want the record where **`status == "accepted"`** and **`taxonRankID == 220`**.
If none of the returned records are "accepted" status, then the queried name is a synonym,
and we read `valid_AphiaID` to find the accepted name.

In [3]:
# Pull out the accepted species-level record
records = resp.json()

accepted = None
for rec in records:
    if rec["taxonRankID"] == 220 and rec["status"] == "accepted":
        accepted = rec
        break

print("AphiaID     :", accepted["AphiaID"])
print("name        :", accepted["scientificname"])
print("status      :", accepted["status"])
print("valid_AphiaID:", accepted["valid_AphiaID"])
print("kingdom     :", accepted["kingdom"])
print("phylum      :", accepted["phylum"])
print("class       :", accepted["class"])

AphiaID     : 495077
name        : Zostera marina
status      : accepted
valid_AphiaID: 495077
kingdom     : Plantae
phylum      : Tracheophyta
class       : Magnoliopsida


When `status` is `"accepted"`, the `AphiaID` and `valid_AphiaID` are the same -- the record
points to itself as the accepted name.

### Not found

When no records match, WoRMS returns **HTTP 204 No Content** -- an empty body with no JSON.
We need to handle this case rather than trying to parse the response.

In [4]:
resp_missing = requests.get(
    f"{BASE_URL}/AphiaRecordsByName/Totally fakeus",
    params={"like": "false", "marine_only": "false", "offset": 1},
)
print("HTTP status:", resp_missing.status_code)
print("Body       :", repr(resp_missing.text))

HTTP status: 204
Body       : ''


---
## 2. Accepted vs. synonym

WoRMS stores both accepted names and their synonyms as separate records, each with its own AphiaID.
When we query a synonym, the response record will have `status` set to `"unaccepted"` and
`valid_AphiaID` pointing to the accepted name.

### 2a. Querying an accepted name

For *Zostera marina*, the record comes back as `status="accepted"` and `AphiaID == valid_AphiaID`.

In [5]:
# Accepted name -- AphiaID and valid_AphiaID are the same
records = requests.get(
    f"{BASE_URL}/AphiaRecordsByName/Zostera marina",
    params={"like": "false", "marine_only": "false", "offset": 1},
).json()

# Only look at species-level records
species_records = [r for r in records if r["taxonRankID"] == 220]

for rec in species_records:
    print(f"AphiaID={rec['AphiaID']}  valid_AphiaID={rec['valid_AphiaID']}  status={rec['status']!r}")

AphiaID=495077  valid_AphiaID=495077  status='accepted'
AphiaID=145795  valid_AphiaID=495077  status='alternative representation'


### 2b. Querying a synonym

*Macrocystis integrifolia* (formerly treated as a separate species of giant kelp) is now considered
a synonym of *Macrocystis pyrifera*. Querying it by name returns `status="unaccepted"`
and `valid_AphiaID` points to the accepted species.

In [6]:
# Synonym -- AphiaID != valid_AphiaID; status is "unaccepted"
records = requests.get(
    f"{BASE_URL}/AphiaRecordsByName/Macrocystis integrifolia",
    params={"like": "false", "marine_only": "false", "offset": 1},
).json()

species_records = [r for r in records if r["taxonRankID"] == 220]

for rec in species_records:
    print(f"AphiaID={rec['AphiaID']}  valid_AphiaID={rec['valid_AphiaID']}  status={rec['status']!r}")
    print(f"  queried name  : {rec['scientificname']}")
    print(f"  accepted name : {rec['valid_name']}")

AphiaID=372509  valid_AphiaID=232231  status='unaccepted'
  queried name  : Macrocystis integrifolia
  accepted name : Macrocystis pyrifera


So the lookup chain for a synonym is:

```
"Macrocystis integrifolia"
  -> AphiaRecordsByName: AphiaID=372509, status="unaccepted", valid_AphiaID=232231
  -> use valid_AphiaID=232231 to get synonyms for "Macrocystis pyrifera"
```

Both paths (accepted name or synonym as input) lead us to the same `valid_AphiaID` to use
in the next step. For an accepted species the `valid_AphiaID` is just its own `AphiaID`.

---
## 3. `AphiaSynonymsByAphiaID/{id}` -- Get synonyms

Given a `valid_AphiaID` (the accepted taxon's ID), this endpoint returns every name in WoRMS
that maps to that accepted taxon. The list includes names at all ranks: species, variety, forma, etc.

We only want species-level synonyms, so we filter by **`taxonRankID == 220`**.

In [7]:
# Get all synonyms for Macrocystis pyrifera (AphiaID 232231)
resp = requests.get(f"{BASE_URL}/AphiaSynonymsByAphiaID/232231")
synonyms_raw = resp.json()

print(f"Total records returned : {len(synonyms_raw)}")
print()

# Show the rank breakdown
from collections import Counter
rank_counts = Counter(r["rank"] for r in synonyms_raw)
for rank, count in rank_counts.items():
    print(f"  {rank}: {count}")

Total records returned : 18

  Species: 15
  Variety: 3


The response mixes species-level names with infraspecific names (Variety, Forma, etc.).
We filter by `taxonRankID == 220` to keep only species-level synonyms.

In [8]:
# Filter to species-level only
species_synonyms = [r for r in synonyms_raw if r["taxonRankID"] == 220]

print(f"Species-level synonyms ({len(species_synonyms)} total):")
for rec in species_synonyms:
    print(f"  {rec['scientificname']}  ({rec['status']})")

Species-level synonyms (15 total):
  Fucus giganteus  (unaccepted)
  Fucus pyriferus  (unaccepted)
  Laminaria pyrifera  (unaccepted)
  Macrocystis angustifolia  (unaccepted)
  Macrocystis communis  (unaccepted)
  Macrocystis humboldtii  (unaccepted)
  Macrocystis integrifolia  (unaccepted)
  Macrocystis laevis  (unaccepted)
  Macrocystis latifolius  (unaccepted)
  Macrocystis luxurians  (unaccepted)
  Macrocystis orbigniana  (unaccepted)
  Macrocystis pelagica  (unaccepted)
  Macrocystis planicaulis  (unaccepted)
  Macrocystis pomifera  (unaccepted)
  Macrocystis tenuifolia  (unaccepted)


The `status` field on each synonym record will typically be `"unaccepted"` for synonyms.
Some may show `"junior subjective synonym"` or similar -- these are all names we want to include.

### Synonym list for an accepted species with fewer synonyms

*Zostera marina* has a smaller synonym list than the giant kelp, which makes it easier to inspect.

In [9]:
# Synonyms for Zostera marina (AphiaID 495077)
resp = requests.get(f"{BASE_URL}/AphiaSynonymsByAphiaID/495077")
synonyms_raw = resp.json()

species_synonyms = [r for r in synonyms_raw if r["taxonRankID"] == 220]

print(f"Total records  : {len(synonyms_raw)}")
print(f"Species-level  : {len(species_synonyms)}")
print()
for rec in species_synonyms:
    print(f"  {rec['scientificname']}  ({rec['status']})")

Total records  : 4
Species-level  : 3

  Zostera angustifolia  (junior subjective synonym)
  Zostera hornemanniana  (unaccepted)
  Zostera marina  (alternative representation)


### No synonyms

When there are no synonyms, WoRMS returns **HTTP 204 No Content** -- same empty response
as when a name is not found. We check the status code before trying to parse JSON.

In [10]:
# Fucus vesiculosus (bladderwrack, AphiaID 145548) has only infraspecific synonyms
# at the species rank -- let's check
resp = requests.get(f"{BASE_URL}/AphiaSynonymsByAphiaID/145548")
print("HTTP status:", resp.status_code)

if resp.status_code == 200:
    synonyms_raw = resp.json()
    species_synonyms = [r for r in synonyms_raw if r["taxonRankID"] == 220]
    print(f"Total records  : {len(synonyms_raw)}")
    print(f"Species-level  : {len(species_synonyms)}")
    for rec in species_synonyms:
        print(f"  {rec['scientificname']}")
else:
    print("No synonyms found (204 No Content)")

HTTP status: 200
Total records  : 50
Species-level  : 4
  Fucus balticus
  Fucus divaricatus
  Fucus excisus
  Fucus inflatus


---
## 4. `AphiaClassificationByAphiaID/{id}` -- Full classification

This endpoint returns the full taxonomic tree for a species, from the top-level grouping
down to the species itself. The structure is a nested object where each node has a `child` key.

This is useful for understanding what group a species belongs to and for filtering to relevant
kingdoms or phyla.

In [11]:
# Classification for Zostera marina (AphiaID 495077)
resp = requests.get(f"{BASE_URL}/AphiaClassificationByAphiaID/495077")
tree = resp.json()

# Walk the nested tree and print each level
node = tree
while node:
    print(f"  {node['rank']:30s}  {node['scientificname']}")
    node = node.get("child")

  Superdomain                     Biota
  Kingdom                         Plantae
  Subkingdom                      Viridiplantae
  Infrakingdom                    Streptophyta
  Phylum (Division)               Tracheophyta
  Subphylum (Subdivision)         Spermatophytina
  Class                           Magnoliopsida
  Superorder                      Lilianae
  Order                           Alismatales
  Family                          Zosteraceae
  Genus                           Zostera
  Species                         Zostera marina


---
## Putting it all together

Here is the full lookup flow we will use in the `get_worms_synonyms` function:

1. Call `AphiaRecordsByName/{name}` to get records for the species name.
2. Filter to species-level records (`taxonRankID == 220`).
3. If no records come back (HTTP 204), the species is not in WoRMS -- return an empty dict.
4. Read `valid_AphiaID` and `valid_name` from the first species record (same value across all records for the same name).
5. Call `AphiaSynonymsByAphiaID/{valid_AphiaID}` to get all linked names.
6. Filter the synonyms to species-level (`taxonRankID == 220`) and collect their `scientificname` values.
7. Build the result: queried name first, then the accepted name if the input was a synonym, then all other synonyms.

Including the accepted name in step 7 means the lookup works both ways -- searching by either
the accepted name or any synonym leads to the same full set of names.

In [12]:
def get_worms_synonyms(species_name: str) -> dict:
    """
    Return a dict of species-level synonym names from WoRMS for the given species name.

    Keys are the queried name plus all species-level synonyms found.
    Values are empty lists (matching the format used by other synonym functions in this project).
    Returns an empty dict if the name is not found in WoRMS.

    When the queried name is itself a synonym, the accepted name is included in the result
    so that lookups work both ways.
    """
    if not species_name or not species_name.strip():
        return {}

    # Step 1-2: look up the name and get species-level records
    resp = requests.get(
        f"{BASE_URL}/AphiaRecordsByName/{species_name.strip()}",
        params={"like": "false", "marine_only": "false", "offset": 1},
    )
    if resp.status_code == 204 or not resp.text:
        return {}
    resp.raise_for_status()

    species_records = [r for r in resp.json() if r["taxonRankID"] == 220]
    if not species_records:
        return {}

    # Step 3-4: get the valid AphiaID and accepted name from the first species record
    valid_aphia_id = species_records[0]["valid_AphiaID"]
    valid_name = species_records[0]["valid_name"]

    # Step 5: fetch synonyms
    syn_resp = requests.get(f"{BASE_URL}/AphiaSynonymsByAphiaID/{valid_aphia_id}")

    synonym_names = []
    if syn_resp.status_code == 200 and syn_resp.text:
        synonym_names = [
            r["scientificname"]
            for r in syn_resp.json()
            if r["taxonRankID"] == 220
        ]

    # Step 6: build result, deduplicated, queried name first
    seen = {species_name.strip()}
    result = [species_name.strip()]
    # When the queried name is a synonym, add the accepted name so lookups work both ways
    if valid_name != species_name.strip() and valid_name not in seen:
        seen.add(valid_name)
        result.append(valid_name)
    for name in synonym_names:
        if name not in seen:
            seen.add(name)
            result.append(name)

    return {name: [] for name in result}

### Accepted species with multiple synonyms

In [13]:
# Cross-check: https://www.marinespecies.org/aphia.php?p=taxdetails&id=232231
get_worms_synonyms("Macrocystis pyrifera")

{'Macrocystis pyrifera': [],
 'Fucus giganteus': [],
 'Fucus pyriferus': [],
 'Laminaria pyrifera': [],
 'Macrocystis angustifolia': [],
 'Macrocystis communis': [],
 'Macrocystis humboldtii': [],
 'Macrocystis integrifolia': [],
 'Macrocystis laevis': [],
 'Macrocystis latifolius': [],
 'Macrocystis luxurians': [],
 'Macrocystis orbigniana': [],
 'Macrocystis pelagica': [],
 'Macrocystis planicaulis': [],
 'Macrocystis pomifera': [],
 'Macrocystis tenuifolia': []}

### Input is itself a synonym

*Macrocystis integrifolia* is a synonym of *Macrocystis pyrifera*. The function resolves to
the accepted taxon first, then returns all synonyms of *Macrocystis pyrifera*. The queried
name appears first, followed by the accepted name, then the remaining synonyms. This matches
the same synonym list as querying *Macrocystis pyrifera* directly, with any duplicate
(the queried name itself, which is already in the list) removed.

In [14]:
# Cross-check: https://www.marinespecies.org/aphia.php?p=taxdetails&id=372509
get_worms_synonyms("Macrocystis integrifolia")

{'Macrocystis integrifolia': [],
 'Macrocystis pyrifera': [],
 'Fucus giganteus': [],
 'Fucus pyriferus': [],
 'Laminaria pyrifera': [],
 'Macrocystis angustifolia': [],
 'Macrocystis communis': [],
 'Macrocystis humboldtii': [],
 'Macrocystis laevis': [],
 'Macrocystis latifolius': [],
 'Macrocystis luxurians': [],
 'Macrocystis orbigniana': [],
 'Macrocystis pelagica': [],
 'Macrocystis planicaulis': [],
 'Macrocystis pomifera': [],
 'Macrocystis tenuifolia': []}

### Species with a small synonym list

In [15]:
# Cross-check: https://www.marinespecies.org/aphia.php?p=taxdetails&id=495077
get_worms_synonyms("Zostera marina")

{'Zostera marina': [], 'Zostera angustifolia': [], 'Zostera hornemanniana': []}

### Species not found in WoRMS

In [16]:
get_worms_synonyms("Totally fakeus speciesname")

{}